# 02 — Feature Engineering
Builds and validates the sklearn preprocessing pipeline used in training and serving.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold

from src.preprocessing import load_and_clean, split_feature_columns, get_feature_columns
from src.features import build_preprocessor, get_feature_names, save_preprocessor

sns.set_theme(style='whitegrid')

## 1. Load cleaned data

In [ ]:
df = load_and_clean('../data/raw/train.csv')
print(f'Shape: {df.shape}')
print(f'missing_count stats:\n{df["missing_count"].describe()}')

## 2. Feature column groups

In [ ]:
col_groups = split_feature_columns(df)
print(f"Categorical (_cat): {col_groups['cat']}")
print(f"\nBinary     (_bin): {col_groups['bin']}")
print(f"\nContinuous  (rest): {col_groups['cont']}")

## 3. Validate pipeline on one CV fold

In [ ]:
feature_cols = get_feature_columns(df)
X = df[feature_cols]
y = df['target']

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, val_idx = next(iter(skf.split(X, y)))

X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

preprocessor = build_preprocessor(df)
X_tr_enc = preprocessor.fit_transform(X_tr, y_tr)
X_val_enc = preprocessor.transform(X_val)

print(f'Train encoded shape: {X_tr_enc.shape}')
print(f'Val   encoded shape: {X_val_enc.shape}')
print(f'Any NaN in train: {np.isnan(X_tr_enc).any()}')
print(f'Any NaN in val:   {np.isnan(X_val_enc).any()}')

## 4. Target encoding sanity check (no leakage)

In [ ]:
# _cat columns should encode into float (target mean per category)
cat_cols = col_groups['cat']
n_cat = len(cat_cols)
if n_cat > 0:
    # First n_cat columns in output correspond to _cat features
    cat_enc = X_tr_enc[:, :n_cat]
    print(f'Cat encoded min={cat_enc.min():.4f}  max={cat_enc.max():.4f}')
    print('Values should be near the global target rate (~0.036) for rare categories')
    print(f'Global target rate: {y_tr.mean():.4f}')

## 5. Distribution of encoded continuous features

In [ ]:
cat_n = len(col_groups['cat'])
cont_n = len(col_groups['cont'])
cont_enc = X_tr_enc[:, cat_n:cat_n + cont_n]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for i, ax in enumerate(axes.flat):
    if i < cont_n and i < 8:
        col_name = col_groups['cont'][i]
        ax.hist(cont_enc[:, i], bins=50, color='steelblue', alpha=0.7)
        ax.set_title(col_name, fontsize=8)
        ax.set_xlabel('Scaled value')
    else:
        ax.set_visible(False)

plt.suptitle('Standardised continuous features (first 8)', y=1.01)
plt.tight_layout()
plt.savefig('../data/feat_cont_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Save fitted preprocessor

In [ ]:
# Fit on full training data for local model artifact
full_preprocessor = build_preprocessor(df)
full_preprocessor.fit(X, y)
save_preprocessor(full_preprocessor, '../models/preprocessor.pkl')
print('Saved: ../models/preprocessor.pkl')

## Summary
- Pipeline handles NaN imputation, target encoding of categoricals, and standard scaling.
- No data leakage: TargetEncoder is fit only on the training fold inside each CV split.
- Output shape: (n_samples, ~43 features).

**Next:** `03_local_training.ipynb` — full CV training and validation.